# HTTPExceptions and Exception Handlers

This notebook covers:

1. The `HTTPException` shape and where it shows up in `/docs`
2. Mapping **domain** errors (raised deep in business logic) to HTTP status codes at the framework edge
3. Registering custom handlers with `@app.exception_handler(...)`
4. A **consistent error response schema** — every error has the same shape
5. Overriding the built-in validation-error handler to match that schema
6. The catch-all 500 handler — what unhandled exceptions look like, and what they should look like

**Scope**: FastAPI + Pydantic v2 + `TestClient`. By the end of the notebook every error — raised from a handler, a dependency, validation, or an uncaught bug — speaks the same JSON dialect.

## 1. The `HTTPException` Pattern

`HTTPException` is the framework-native way to short-circuit a request with a non-2xx response. Raise it from anywhere a `Depends` or handler runs and FastAPI turns it into a JSON body for you:

```python
raise HTTPException(status_code=404, detail="Asset not found")
```

Three properties to lock in:

- **`status_code`** is the HTTP status; FastAPI sets the response status to it.
- **`detail`** is the body. It can be a string (`{"detail": "..."}`), a dict (`{"detail": {...}}`), or even a list. We'll standardize on a dict envelope in section 4.
- **`headers`** lets you attach response headers — useful for `WWW-Authenticate: Bearer` on 401 (notebook 5.1) or `Retry-After` on 429 (notebook 5.3).

Use `HTTPException` for *expected* error states ("the asset doesn't exist," "the user already exists," "the request violates a business rule"). Reserve uncaught exceptions for *unexpected* states (a `KeyError` in a code path you didn't anticipate) — those become 500s, covered in section 6.

In [ ]:
from fastapi import FastAPI, HTTPException, status
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

class Asset(BaseModel):
    ticker: str = Field(pattern=r"^[A-Z.]{1,10}$")
    name: str
    price: float = Field(ge=0)

ASSETS_SEED: dict[str, Asset] = {
    "AAPL": Asset(ticker="AAPL", name="Apple Inc.", price=190.0),
    "MSFT": Asset(ticker="MSFT", name="Microsoft Corp.", price=420.0),
}

demo_app = FastAPI()

@demo_app.get("/assets/{ticker}", response_model=Asset)
def get_asset(ticker: str):
    ticker = ticker.upper()
    asset = ASSETS_SEED.get(ticker)
    if asset is None:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail=f"Asset '{ticker}' not found",
        )
    return asset

demo_client = TestClient(demo_app)
print("hit       :", demo_client.get("/assets/AAPL").json())
miss = demo_client.get("/assets/NVDA")
print("miss      :", miss.status_code, miss.json())

Notice the default error body shape: `{"detail": "Asset 'NVDA' not found"}`. That's FastAPI's built-in serialization for `HTTPException` and it's *fine* for a one-app project. The moment you have two services, or a frontend that wants to render errors consistently, you'll wish the body always looked the same regardless of which kind of error was raised — that's section 4.

## 2. Mapping Domain Errors to HTTP

Business logic shouldn't know about HTTP. A repository function (notebook 4.2) raising `HTTPException(404)` couples it to FastAPI; the same function can't be reused in a CLI script or a worker. The clean shape:

- **Domain exceptions** live in the business-logic layer: `AssetNotFound`, `DuplicateTicker`, `InsufficientFunds`. Plain Python, no FastAPI import.
- **The framework edge** (handlers, exception handlers) translates each domain exception to a status code and response body.

That separation lets the same `AssetRepo.get(ticker)` raise `AssetNotFound` whether it's called from an HTTP route, a Celery task, or a unit test — only the HTTP layer cares about 404.

In [ ]:
# Domain layer — no FastAPI import. Could live in src/portfolio/exceptions.py.
class DomainError(Exception):
    """Base class so the HTTP layer can register one handler that catches the whole family."""

class AssetNotFound(DomainError):
    def __init__(self, ticker: str):
        self.ticker = ticker
        super().__init__(f"Asset '{ticker}' not found")

class DuplicateTicker(DomainError):
    def __init__(self, ticker: str):
        self.ticker = ticker
        super().__init__(f"Ticker '{ticker}' already exists")

class MarketClosed(DomainError):
    """Raised by an order-placement service when the market isn't open."""

class AssetService:
    """Business logic — uses domain exceptions, NOT HTTPException."""
    def __init__(self, store: dict[str, Asset]):
        self.store = store

    def get(self, ticker: str) -> Asset:
        ticker = ticker.upper()
        asset = self.store.get(ticker)
        if asset is None:
            raise AssetNotFound(ticker)
        return asset

    def create(self, asset: Asset) -> Asset:
        if asset.ticker in self.store:
            raise DuplicateTicker(asset.ticker)
        self.store[asset.ticker] = asset
        return asset

# Quick check — the service raises domain exceptions, period.
svc = AssetService(dict(ASSETS_SEED))
try:
    svc.get("NVDA")
except AssetNotFound as e:
    print("caught in pure Python:", type(e).__name__, "|", e)

Why a `DomainError` base class? Section 3 shows: it lets the HTTP layer register **one** handler that catches the entire family. New domain exception → automatically handled, no new wiring.

## 3. Custom Exception Handlers

`@app.exception_handler(ExcType)` registers a function that turns `ExcType` (or any subclass) into a `Response`. FastAPI invokes it whenever that exception escapes from a handler or dependency. The function gets `(request, exception)` and returns a `Response` (most commonly a `JSONResponse` with the chosen status code).

The pattern: one handler per *category*, keyed off a common base class. Inside the handler, branch on the concrete type to pick the status code.

In [ ]:
from fastapi import Request
from fastapi.responses import JSONResponse

# Map every domain exception type -> HTTP status. One source of truth.
DOMAIN_STATUS = {
    AssetNotFound:   status.HTTP_404_NOT_FOUND,
    DuplicateTicker: status.HTTP_409_CONFLICT,
    MarketClosed:    status.HTTP_503_SERVICE_UNAVAILABLE,
}

v3_app = FastAPI()
v3_service = AssetService(dict(ASSETS_SEED))

@v3_app.exception_handler(DomainError)
async def handle_domain_error(request: Request, exc: DomainError):
    code = DOMAIN_STATUS.get(type(exc), status.HTTP_500_INTERNAL_SERVER_ERROR)
    return JSONResponse(
        status_code=code,
        content={"detail": str(exc), "code": type(exc).__name__},
    )

@v3_app.get("/assets/{ticker}", response_model=Asset)
def get_asset(ticker: str):
    # No try/except, no HTTPException — the service raises a domain error and
    # the handler above translates it. Routes stay readable.
    return v3_service.get(ticker)

@v3_app.post("/assets", response_model=Asset, status_code=status.HTTP_201_CREATED)
def create_asset(asset: Asset):
    return v3_service.create(asset)

c3 = TestClient(v3_app)
print("hit              :", c3.get("/assets/AAPL").status_code)
miss = c3.get("/assets/NVDA")
print("miss             :", miss.status_code, miss.json())
dup = c3.post("/assets", json={"ticker": "AAPL", "name": "dup", "price": 1})
print("duplicate ticker :", dup.status_code, dup.json())

Notice what the route functions don't have anymore: no `if asset is None: raise HTTPException(...)` boilerplate. The route reads as the business operation it represents. Adding a new domain exception — say, `InsufficientFunds` — means **one** line in `DOMAIN_STATUS`. No handler wiring per route.

Two design tradeoffs:

- **A central `DOMAIN_STATUS` dict** scales well up to ~30 exceptions. Past that, push the status code onto the exception class itself (`AssetNotFound.http_status = 404`) so each exception is self-describing.
- **One handler per family, not per type.** Don't register a separate `@app.exception_handler(AssetNotFound)`, `@app.exception_handler(DuplicateTicker)`, etc. The base-class handler is one function; per-type handlers fan out the wiring.

## 4. A Consistent Error-Response Shape

The two responses we've returned so far don't quite match:

- A direct `HTTPException`: `{"detail": "Asset 'NVDA' not found"}`
- Our domain handler: `{"detail": "...", "code": "AssetNotFound"}`
- Validation errors (next section): a *list* under `detail`.

Three different shapes is exactly what client code hates. Let's pick one envelope and enforce it across **every** error path. The schema we'll commit to:

```json
{
  "error": {
    "code": "AssetNotFound",
    "message": "Asset 'NVDA' not found",
    "request_id": "req_abc123",
    "details": {...}
  }
}
```

- **`code`** — machine-readable enum the client can branch on. *Don't* compare error messages; they'll change.
- **`message`** — human-readable, safe to surface in UI or logs.
- **`request_id`** — populated by the request-ID middleware in 6.2; lets the client quote it to support and you grep your logs.
- **`details`** — structured per-field info (most useful on validation errors).

To keep the demo straightforward, the rest of the notebook uses **one** app with all four handlers registered up front:

In [ ]:
from typing import Any
import logging
from fastapi.exceptions import RequestValidationError

class ErrorBody(BaseModel):
    code: str
    message: str
    request_id: str | None = None
    details: dict[str, Any] | None = None

class ErrorResponse(BaseModel):
    error: ErrorBody

def error_response(*, code: str, message: str, status_code: int,
                   request_id: str | None = None,
                   details: dict | None = None) -> JSONResponse:
    payload = ErrorResponse(error=ErrorBody(
        code=code, message=message, request_id=request_id, details=details,
    ))
    return JSONResponse(status_code=status_code, content=payload.model_dump(exclude_none=True))

logger = logging.getLogger("uvicorn.error")  # same logger Uvicorn uses for unhandled exceptions

app = FastAPI()
service = AssetService(dict(ASSETS_SEED))

# Handler #1 — every domain exception, mapped through DOMAIN_STATUS.
@app.exception_handler(DomainError)
async def on_domain_error(request: Request, exc: DomainError):
    return error_response(
        code=type(exc).__name__,
        message=str(exc),
        status_code=DOMAIN_STATUS.get(type(exc), 500),
        request_id=request.headers.get("x-request-id"),
    )

# Handler #2 — any HTTPException raised directly from a route gets the same envelope.
@app.exception_handler(HTTPException)
async def on_http_exception(request: Request, exc: HTTPException):
    if isinstance(exc.detail, dict) and "code" in exc.detail:
        code, message = exc.detail["code"], exc.detail.get("message", "")
    else:
        code, message = f"HTTP_{exc.status_code}", str(exc.detail)
    return error_response(
        code=code, message=message, status_code=exc.status_code,
        request_id=request.headers.get("x-request-id"),
    )

# Handler #3 — Pydantic validation failures, repackaged into the envelope.
@app.exception_handler(RequestValidationError)
async def on_validation_error(request: Request, exc: RequestValidationError):
    field_errors = [
        {"loc": list(err["loc"]), "type": err["type"], "msg": err["msg"]}
        for err in exc.errors()
    ]
    return error_response(
        code="ValidationError",
        message="Request validation failed",
        status_code=status.HTTP_422_UNPROCESSABLE_ENTITY,
        request_id=request.headers.get("x-request-id"),
        details={"fields": field_errors},
    )

# Handler #4 — last resort: any Exception we didn't model. Log the traceback, body is vague.
@app.exception_handler(Exception)
async def on_unhandled(request: Request, exc: Exception):
    logger.exception("unhandled exception on %s %s", request.method, request.url.path)
    return error_response(
        code="InternalError",
        message="An unexpected error occurred. Please retry; if it persists, contact support with the request id.",
        status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,
        request_id=request.headers.get("x-request-id"),
    )

# Routes for the demo — same domain layer, exercising each handler.
@app.get("/assets/{ticker}", response_model=Asset)
def get_asset(ticker: str):
    return service.get(ticker)               # may raise AssetNotFound

@app.post("/assets", response_model=Asset, status_code=201)
def create_asset(asset: Asset):
    return service.create(asset)             # may raise DuplicateTicker; body may 422

@app.get("/manual-404")
def manual_404():
    raise HTTPException(status_code=404, detail={"code": "NoSuchRoute", "message": "Not here"})

@app.get("/oops")
def oops():
    return 1 / 0                             # ZeroDivisionError -> on_unhandled -> 500

# raise_server_exceptions=False so TestClient returns the 500 body instead of re-raising.
client = TestClient(app, raise_server_exceptions=False)
print("domain miss   :", client.get("/assets/NVDA").json())
print("manual http   :", client.get("/manual-404").json())

Every body now has `{"error": {"code": ..., "message": ...}}`. The frontend can `switch (response.error.code)` without parsing free-text. Logs can be filtered by code. Alerts can fire on counts of `"AssetNotFound"` separately from `"DuplicateTicker"`. That's the *point* of a stable error code — it's a contract the client and your monitoring both depend on.

## 5. Validation Errors (422)

Pydantic validation failures produce `RequestValidationError` (a subclass of Pydantic's `ValidationError`). FastAPI's default handler emits 422 with a list of per-field errors:

```json
{"detail": [{"type": "missing", "loc": ["body", "ticker"], "msg": "..."}]}
```

The information is excellent; the shape doesn't match our envelope. We overrode `RequestValidationError` in the cell above; here's the result. Force a 422 by violating two constraints at once: bad ticker pattern + negative price.

In [ ]:
import json
bad = client.post("/assets", json={"ticker": "lower", "name": "", "price": -1})
print("status:", bad.status_code)
print(json.dumps(bad.json(), indent=2))

Same envelope. The field-by-field breakdown lives under `error.details.fields` where a UI form can read it and render per-input messages — but the *outer* shape is identical to every other error response, so generic client code ("if 4xx, log and show the message") still works.

Three things to keep in mind for production:

- **Don't expose validator internals in messages.** Pydantic's default messages are great for developers and noisy for end users. If you serve a public frontend, map the `type` (e.g., `"string_pattern_mismatch"`) to a friendly message at the frontend layer rather than in the response.
- **Don't dump the entire `errors()` list.** It can include the rejected value — fine for development, leaky for a public API if a user pasted something secret into the request.
- **OpenAPI documents 422 automatically** for any route with a body schema. Pin the response example to your envelope by setting `responses={422: {"model": ErrorResponse}}` on routes that should show the unified shape in `/docs`.

## 6. The Catch-All 500 Handler

The errors so far are *expected*: validation, missing assets, duplicate keys. They have a code, a status, a message. The unexpected ones — a `KeyError` from a malformed JSON parse, a `ZeroDivisionError` in a pricing calculation, a database connection drop — also need to leave a clean trail.

Without a handler, FastAPI returns Starlette's default 500: `Internal Server Error` as plain text. Three problems:

- The body shape doesn't match the envelope clients depend on.
- The stack trace might leak into logs without a request ID attached.
- The status is `500` with no `code` — alerts can't differentiate "all 500s are the same" from "only 500s with code X spiked."

We registered `@app.exception_handler(Exception)` in the section-4 cell. Let's exercise it. Note that `TestClient` defaults to **re-raising** server-side exceptions in tests; we constructed `client` with `raise_server_exceptions=False` so we can inspect the actual HTTP response a real client would see.

In [ ]:
r = client.get("/oops")
print("status :", r.status_code)
print("body   :", r.json())

Three properties that make this 500 handler safe to ship:

- **The full traceback goes to logs**, not the response. `logger.exception(...)` captures the stack via Python's logging system; the response body is intentionally vague.
- **The message has no exception details.** No `"division by zero"`, no file paths, no internal symbol names. Those are an attacker's reconnaissance and a maintenance burden once a client hardcodes them.
- **The `request_id` is the bridge.** A user reports "got an error at 3:42 PM," quotes `req_abc123`, and you grep your logs for that ID and find the traceback. The request ID is the only piece of correlation data the body carries — and section 6.2 is where we populate it.

The ordering of handlers matters: FastAPI tries the most specific class first. `RequestValidationError` → `DomainError` → `HTTPException` → `Exception` (last resort). If you register them all, validation errors won't accidentally hit the 500 handler — the specific one wins.

## Key Takeaways

- **`HTTPException`** is for short-circuit responses with a status code. Use it from routes; reserve it for cases the route can't avoid.
- **Keep business logic free of `HTTPException`.** Raise plain domain exceptions (`AssetNotFound`, `DuplicateTicker`) from services and repositories so they're reusable outside HTTP.
- **`@app.exception_handler(BaseDomainError)`** turns the whole family into HTTP at the framework edge. One handler, one `DOMAIN_STATUS` mapping — adding a new exception type is one line.
- **One error envelope: `{error: {code, message, request_id, details}}`.** Codes are machine-readable; messages are human-readable; request IDs are the audit thread that ties body to log.
- **Override `RequestValidationError`** to put 422 in the envelope; expose per-field issues under `details.fields`.
- **Register an `Exception` handler** as the last resort. Log the traceback; never leak it. The body says "please contact support with the request ID."
- **Handler ordering** is by specificity: validation → domain → HTTPException → Exception. Wire all four and every response is consistent.
- **Capstone tie-in**: `exceptions.py` will host the `DomainError` hierarchy; `main.py` will register all four handlers; the envelope above is what every client sees, regardless of which layer raised.

## Exercises

**1. Unify two diverging error shapes.** Imagine two routes you wrote yesterday: one raises `HTTPException(404, detail="asset not found")`, the other raises `HTTPException(404, detail={"code": "AssetMissing", "message": "asset not found"})`. The frontend has to branch on shape. The `on_http_exception` handler above already normalizes *both* into the `{error: {code, message}}` envelope — verify with TestClient that both routes now return identical structure, then try replacing the dict-detail route with a string-detail one and confirm the envelope is unchanged.

**2. Add a new domain exception with zero route changes.** Add `class InsufficientFunds(DomainError)` raised from a `place_order(user, asset, qty)` service method. Wire it into `DOMAIN_STATUS` as 402 (Payment Required, fittingly). Add a `POST /orders` route. Verify the response body matches the envelope and `code="InsufficientFunds"`. Note how many lines you had to change in the route handler. (Answer: zero.)

**3. Per-route 422 documentation.** Add `responses={422: {"model": ErrorResponse}}` to your `POST /assets` route. Inspect `/openapi.json` (or `/docs`) and confirm the 422 response shows the envelope, not the default FastAPI shape. Now extend to 404 on `GET /assets/{ticker}`. (Hint: `responses=` takes a dict keyed by status code; each value can include a `model`, a description, and content examples.)